In [1]:
!git clone https://github.com/Ahnd6474/Protein-DB

Cloning into 'Protein-DB'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 183 (delta 69), reused 88 (delta 25), pack-reused 26 (from 2)
Receiving objects: 100% (183/183), 61.17 MiB | 42.26 MiB/s, done.
Resolving deltas: 100% (69/69), done.


In [2]:
%cd /kaggle/working/Protein-DB

/kaggle/working/Protein-DB


In [3]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.

In [4]:
from Bio import SeqIO
seq_path='/kaggle/input/uniref50-sub/uniref50_subsample.fasta'
sequences=[]
for seq_record in SeqIO.parse(seq_path, "fasta"):
    sequences.append(str(seq_record.seq))
print(len(sequences))

1000000


In [5]:
# 길이 512 이하인 시퀀스만 남기기
seq = [s for s in sequences if len(s) <= 512]

print(len(seq))

873074


In [6]:
from vae_module import Tokenizer, Config, load_vae, encode
cfg = Config(model_path="models/vae_epoch380.pt")
tok = Tokenizer.from_esm()

model = load_vae(cfg,
                 vocab_size=len(tok.vocab),
                 pad_idx=tok.pad_idx,
                 bos_idx=tok.bos_idx)


2025-10-31 04:14:15 vae_module.loader [INFO] Loaded VAE from models/vae_epoch380.pt on cpu


In [7]:
from tqdm.auto import tqdm
import torch

# 1) 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 2) 프로그레스 바 단일 생성
vec = []
for sequence in tqdm(seq, desc="Encoding sequences", unit="seq", leave=False):
    vec_tensor = encode(model, sequence, tok, cfg.max_len)
    vec.append(vec_tensor.cpu())


Encoding sequences:   0%|          | 0/873074 [00:00<?, ?seq/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
/usr/local/lib/python3.11/dist-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


In [8]:
l = {seqs: vecs for seqs, vecs in zip(seq, vec)}
import numpy as np

import numpy as np
from typing import Dict, Tuple

def _validate_mapping(mapping: Dict[str, np.ndarray]) -> np.ndarray:
    """
    mapping: {sequence(str): embedding(np.ndarray shape (D,))}
    returns X with shape (N, D) stacked in the SAME ORDER as seqs list
    """
    if not mapping:
        raise ValueError("mapping is empty")

    # fix order so it's stable/reproducible: we lock in the insertion order
    seqs = list(mapping.keys())

    # stack all embeddings
    X_list = []
    dim = None
    for s in seqs:
        v = np.asarray(mapping[s])
        if v.ndim != 1:
            raise ValueError(f"embedding for {s} is not 1-D, got shape {v.shape}")
        if dim is None:
            dim = v.shape[0]
        elif v.shape[0] != dim:
            raise ValueError(f"inconsistent dims: expected {dim}, got {v.shape[0]} for {s}")
        X_list.append(v)

    X = np.stack(X_list, axis=0)  # (N, D)
    return seqs, X


def save_npz(mapping: Dict[str, np.ndarray],
             path: str = "/kaggle/working/seq_vec.npz") -> None:
    """
    Save:
      seqs -> array(dtype=object) of length N
      X    -> float array (N, D)
    """
    seqs, X = _validate_mapping(mapping)
    seqs_arr = np.array(seqs, dtype=object)
    np.savez(path, seqs=seqs_arr, X=X)


def load_npz(path: str = "/kaggle/working/seq_vec.npz") -> Tuple[list, np.ndarray]:
    """
    Load:
      returns (seqs_list, X_matrix)
      seqs_list: List[str] length N
      X_matrix : np.ndarray (N, D)
    """
    data = np.load(path, allow_pickle=True)
    seqs = data["seqs"].tolist()  # back to normal python list[str]
    X = data["X"]
    return seqs, X

save_npz(l)